In [1]:
!pip install -e ..

Obtaining file:///home/mohammad/learn/starcraft/co-op/analyzer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for analyzer (pyproject.toml) ... done
  Created wheel for analyzer: filename=analyzer-0.1.0-0.editable-py3-none-any.whl size=1180 sha256=edaa7572d1ecf32920cd00d97c0da66e1f3e985eb4a4bafc43bf554c765a4c46
  Stored in directory: /tmp/pip-ephem-wheel-cache-p08lk5dd/wheels/80/68/03/98f7d7f72c71ba4eef83db7a82855e13aee68cd2747d293bb9
Successfully built analyzer
  Attempting uninstall: analyzer
    Found existing installation: analyzer 0.1.0
    Can't uninstall 'analyzer'. No files were found to uninstall.


In [1]:
from replay.army_processor import ArmyProcessor
import sc2reader

replay1 = sc2reader.load_replay("/home/mohammad/StarCraft II/Accounts/1176921989/2-S2-1-11021412/Replays/commanders/nova/abathur/Chain of Ascension-375.SC2Replay")

processor = ArmyProcessor(replay1)
units, unit_events = processor.process_replay()

In [5]:
from collections import Counter

counter = Counter()

for event in replay1.tracker_events:
    counter[event.name] += 1

print(counter)

Counter({'UnitBornEvent': 4778, 'UnitDiedEvent': 4443, 'UpgradeCompleteEvent': 991, 'UnitTypeChangeEvent': 456, 'PlayerStatsEvent': 301, 'UnitInitEvent': 91, 'UnitDoneEvent': 85, 'UnitPositionsEvent': 66, 'PlayerSetupEvent': 9})


## calculate lifetime

In [3]:
temp_units = processor.ephemeral_units(units, 10)
temp_units

{'BiomassPickup',
 'Broodling',
 'LocustFlying',
 'NovaBoombotBurrowed',
 'SlaynElementalGrabAOEGroundUnit100',
 'SlaynElementalGrabAOEGroundUnit150',
 'SlaynElementalGrabAOEGroundUnit25',
 'SlaynElementalGrabAOEGroundUnit50',
 'SlaynElementalGrabAOEGroundUnit75',
 'ZergDropPod',
 'ZergDropPodCreep'}

In [27]:
import pandas as pd

lifetimes = []

for unit in units.values():

    if unit.death_frame is None:
        continue

    lifetime = unit.death_frame - unit.birth_frame

    lifetimes.append({
        "unit_type": unit.current_type,
        "lifetime": lifetime
    })

df = pd.DataFrame(lifetimes)
df1 = df.groupby("unit_type")["lifetime"].mean()
df1[df1 > 0].sort_values(ascending=True)
#print(df1.sort_values(ascending=True))
#print(df1)

unit_type
NovaGriffinBombingRunTargeter            0.788690
SlaynElementalGrabAOEGroundUnit150       1.364087
SlaynElementalGrabAOEGroundUnit175       1.428571
SlaynElementalGrabAOEGroundUnit50        1.447704
Broodling                                1.838898
                                         ...     
UltraliskCavern                        914.419643
BanelingNest                           953.229167
HydraliskDen                           977.500000
InfestationPit                         979.441964
PitMalash                             1063.437500
Name: lifetime, Length: 67, dtype: float64

In [9]:
types = set()
for u in units.values():
    types.add(u.current_type)
    if u.current_type == "Goliath_BlackOps":
        print(u)

print(types)

Unit(unit_id=24117275, owner=2, current_type='Goliath_BlackOps', is_army=False, is_building=False, is_worker=False, birth_frame=582.9464285714286, death_frame=705.9821428571429, type_history=[(13058, 'Goliath_BlackOps')])
Unit(unit_id=35913747, owner=2, current_type='Goliath_BlackOps', is_army=False, is_building=False, is_worker=False, birth_frame=582.9464285714286, death_frame=None, type_history=[(13058, 'Goliath_BlackOps')])
Unit(unit_id=29622287, owner=2, current_type='Goliath_BlackOps', is_army=False, is_building=False, is_worker=False, birth_frame=621.6964285714286, death_frame=None, type_history=[(13926, 'Goliath_BlackOps')])
Unit(unit_id=59768837, owner=2, current_type='Goliath_BlackOps', is_army=False, is_building=False, is_worker=False, birth_frame=621.6964285714286, death_frame=None, type_history=[(13926, 'Goliath_BlackOps')])
{'Hydralisk', 'Devourer', 'LocustFlying', 'NovaDefensiveMatrixDrone', 'BiomassPickup', 'BanelingBurrowed', 'Marauder_BlackOpsSpawnerUnit', 'HellbatBlac

In [3]:
interesting = {
    "Locust",
    "ToxicNest",
    "NovaBoombot",
    "TychusWarhoundAutoTurret",
}

for event in replay1.tracker_events:
    if (
            hasattr(event, "unit_type_name")
            and event.unit_type_name in interesting
    ):
        print(
            event.unit_type_name,
            getattr(event.unit, "is_army", None),
            getattr(event.unit, "is_building", None)
        )

ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
Locust False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
Locust False False
Locust False False
Locust False False
NovaBoombot False False
Locust False False
Locust False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
Locust False False
ToxicNest False False
Locust False False
ToxicNest False False
Locust False False
Locust False False
Locust False False
NovaBoombot False False
ToxicNest False False
ToxicNest False False
ToxicNest False False
ToxicNest False F

## check if my replay has all needed event
* UnitBornEvent
* UnitInitEvent
* UnitDoneEvent
* UnitTypeChangeEvent
* UnitDiedEvent

In [3]:
from collections import Counter


def debug_event_counts(replay):
    counter = Counter()

    for event in replay.tracker_events:
        counter[event.name] += 1

    return counter

print(debug_event_counts(replay1))

Counter({'UnitBornEvent': 4778, 'UnitDiedEvent': 4443, 'UpgradeCompleteEvent': 991, 'UnitTypeChangeEvent': 456, 'PlayerStatsEvent': 301, 'UnitInitEvent': 91, 'UnitDoneEvent': 85, 'UnitPositionsEvent': 66, 'PlayerSetupEvent': 9})


In [ ]:
from collections import Counter

event_counts = Counter(
    event["action"]
    for event in unit_events
)

print(event_counts)

Counter({'born': 2874, 'died': 2516, 'complete': 85, 'morph': 2})


In [5]:
import pandas as pd


def unit_events_df(unit_events):
    return pd.DataFrame(unit_events)

df = unit_events_df(unit_events)

print(df.head())
print(df["action"].value_counts())

   time action  unit_id      unit_type  player old_type new_type
0     0   born  2621441   SpineCrawler       4      NaN      NaN
1     0   born  3407873  SuperWarpGate       3      NaN      NaN
2     0   born  3670017   SpineCrawler       3      NaN      NaN
3     0   born  3932161   SpineCrawler       3      NaN      NaN
4     0   born  5767169   BanelingNest       6      NaN      NaN
action
born        4780
died        4422
complete      85
morph          2
Name: count, dtype: int64


In [6]:
df[df["action"] == "morph"]
df.groupby(
    ["old_type", "new_type"]
).size().sort_values(ascending=False)

old_type  new_type
Hatchery  Lair        1
Lair      Hive        1
dtype: int64